# Week 46, Day 5: Prefrontal cortex and context: one network, many tasks

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w046/d5.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Train a readout on a random network

Complete run_trials: for each of 40 time steps, build the input (noisy motion, noisy colour, and two context lines unless ctx_input is False), update X = tanh(X W^T + 0.5 u W_in^T), and average X over the last 10 steps. Then fit a ridge readout of the correct answer and test it on new trials, especially on conflict trials where motion and colour point opposite ways.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
r = np.random.default_rng(2013)
N, T = 300, 40
W = 0.9 * r.normal(size=(N, N)) / np.sqrt(N)
W_in = r.normal(size=(N, 4))

def run_trials(n, ctx_input=True):
    cm = r.choice([-1, 1], n) * r.choice([0.05, 0.15, 0.5], n)      # motion coherence
    cc = r.choice([-1, 1], n) * r.choice([0.05, 0.15, 0.5], n)      # colour coherence
    ctx = r.integers(0, 2, n)                                       # 0: report motion, 1: report colour
    X = np.zeros((n, N)); avg = np.zeros((n, N))
    for t in range(T):
        u = ...
        X = ...
        if t >= T - 10:
            avg += X / 10
    y = np.where(ctx == 0, np.sign(cm), np.sign(cc))
    return avg, y, cm, cc, ctx

def ridge(X, y, lam=1.0):
    Xb = np.c_[X, np.ones(len(X))]
    return np.linalg.solve(Xb.T @ Xb + lam * np.eye(Xb.shape[1]), Xb.T @ y)

predict = lambda w, X: np.sign(np.c_[X, np.ones(len(X))] @ w)

Xtr, ytr, *_ = run_trials(3000)
Xte, yte, cm, cc, ctx = run_trials(2000)
w = ridge(Xtr, ytr); p = predict(w, Xte)
conflict = np.sign(cm) != np.sign(cc)
acc_conflict = np.mean(p[conflict] == yte[conflict])
Xtr0, ytr0, *_ = run_trials(3000, ctx_input=False)
Xte0, yte0, cm0, cc0, _ = run_trials(2000, ctx_input=False)
p0 = predict(ridge(Xtr0, ytr0), Xte0); conf0 = np.sign(cm0) != np.sign(cc0)
acc_conflict_noctx = np.mean(p0[conf0] == yte0[conf0])
print(f"overall accuracy {np.mean(p == yte):.3f}; on conflict trials {acc_conflict:.3f}; without context input {acc_conflict_noctx:.3f}")

In [ ]:
import numpy as np
def test_learns():
    "with the context input, the network is above 68% correct on conflict trials"
    assert acc_conflict > 0.68
def test_needs_context():
    "without the context input, it is near chance on conflict trials"
    assert acc_conflict_noctx < 0.58

test_learns()
test_needs_context()
print("All checks passed")

### Exercise: Is the irrelevant feature still there?

Within each context, fit separate ridge decoders for the sign of motion and the sign of colour on half the test trials and score them on the other half. Then compute accuracy in the motion context as a function of the strength of motion (relevant) and of colour (irrelevant).

In [ ]:
dec = {}
for feat, v in [("motion", np.sign(cm)), ("colour", np.sign(cc))]:
    for c in [0, 1]:
        idx = np.where(ctx == c)[0]; h = len(idx) // 2
        wd = ...
        dec[(feat, c)] = np.mean(predict(wd, Xte[idx[h:]]) == v[idx[h:]])
for (feat, c), a_ in dec.items():
    print(f"{feat} decoded in the {'motion' if c == 0 else 'colour'} context: {a_:.3f}")

m0 = ctx == 0
rel = {s: np.mean((p == yte)[m0 & (np.abs(cm) == s)]) for s in [0.05, 0.15, 0.5]}
irr = {s: np.mean((p == yte)[m0 & (np.abs(cc) == s)]) for s in [0.05, 0.15, 0.5]}
print("motion context, accuracy by motion strength:", ", ".join(f"{k}: {v:.2f}" for k, v in rel.items()))
print("motion context, accuracy by colour strength:", ", ".join(f"{k}: {v:.2f}" for k, v in irr.items()))

In [ ]:
import numpy as np
def test_both():
    "motion and colour are each decodable above 62% in both contexts"
    assert min(dec.values()) > 0.62
def test_psycho():
    "choices follow the relevant feature's strength, not the irrelevant one's"
    assert rel[0.5] > rel[0.05] + 0.25 and abs(irr[0.5] - irr[0.05]) < 0.1

test_both()
test_psycho()
print("All checks passed")